# Assignment: When Chat Systems Break - A Realistic Sharding Simulation


---
## 📅 Day 6 (6 April): User-Based Sharding — First Wrong Decision

In [ ]:
import random


class Message:
    def __init__(self, user_id, channel_id, content):
        self.user_id = user_id
        self.channel_id = channel_id
        self.content = content


class Shard:
    def __init__(self, shard_id):
        self.id = shard_id
        self.messages = []

    def store(self, message):
        self.messages.append(message)

    def count(self):
        return len(self.messages)


class ShardManager:
    def __init__(self, num_shards):
        self.shards = [Shard(i) for i in range(num_shards)]

    def send_message(self, message):
        # will be implemented by subclasses
        pass

    def print_stats(self):
        total = sum(s.count() for s in self.shards)
        print(f"{'Shard':<10} {'Messages':<15} {'% of Total':<10}")
        print('-' * 35)
        for shard in self.shards:
            pct = round((shard.count() / total * 100), 1) if total > 0 else 0
            print(f"Shard {shard.id:<5} {shard.count():<15} {pct}%")
        print(f"{'Total':<10} {total}")


In [ ]:
# =============================================
# DAY 6: User-Based Sharding
# Logic: user_id % num_shards → decides which shard
# Problem: one popular user floods one shard
# =============================================

class UserShardManager(ShardManager):

    def get_shard(self, user_id):
        # user 0,3,6 → shard 0 | user 1,4,7 → shard 1 | user 2,5,8 → shard 2
        return self.shards[user_id % len(self.shards)]

    def send_message(self, message):
        shard = self.get_shard(message.user_id)
        shard.store(message)


# --- Simulate: 1 influencer (user_id=1) sends 5000 messages ---
# --- Other 99 normal users send a few messages each ---

print("=== User-Based Sharding Simulation ===")
print("Scenario: 1 influencer (user 1) sends 5000 messages")
print("          99 normal users send ~10 messages each")
print()

user_manager = UserShardManager(num_shards=3)

# Influencer sends 5000 messages
for i in range(5000):
    msg = Message(user_id=1, channel_id=random.randint(1, 10), content="watch my stream!")
    user_manager.send_message(msg)

# 99 normal users each send ~10 messages
for user in range(2, 101):
    for _ in range(10):
        msg = Message(user_id=user, channel_id=random.randint(1, 10), content="hello")
        user_manager.send_message(msg)

print("--- Messages per Shard ---")
user_manager.print_stats()
print()
print("--- Observation ---")
print("User 1 → user_id 1 % 3 = Shard 1")
print("Shard 1 is OVERLOADED with 5000 messages from just 1 user!")
print("Shards 0 and 2 are mostly idle.")
print("This is called LOAD IMBALANCE — one shard doing all the work.")